# Fermion-to-qubit encoding: FermionOperator + Jordan-Wigner

Walkthrough of `core/fermions.py` (Task 20). The point of this module is that a fermionic model is defined once, encoding-agnostic, and only becomes qubits when you explicitly ask via `.encode(...)`.

In [1]:
import numpy as np

from qaravan.core.fermions import (
    FermionOperator,
    FermionicEncoding,
    hopping,
    majorana_bilinear,
)
from qaravan.core.hamiltonians import FermiHubbard1D, interleaved_mode_order
from qaravan.core.observables import PauliString

np.set_printoptions(precision=4, suppress=True)

## 1. A bare fermionic Hamiltonian: just hopping terms

`FermionOperator` is pure algebra -- a sum of `FermionTerm`s (products of ladder operators), with no notion of qubits at all. `hopping(a, b, coeff)` returns the two `FermionTerm`s making up `coeff * (c†_a c_b + c†_b c_a)`.

Here's a 4-mode chain with nearest-neighbor hopping only (no interaction, no chemical potential) -- deliberately simpler than `FermiHubbard1D`, to see the bare mechanism before the model-specific machinery.

In [2]:
num_modes = 4
t = 1.0

hopping_terms = []
for site in range(num_modes - 1):
    hopping_terms += hopping(site, site + 1, -t)
hopping_terms

[FermionTerm(coeff=-1.0, ops=[(0, True), (1, False)]),
 FermionTerm(coeff=-1.0, ops=[(1, True), (0, False)]),
 FermionTerm(coeff=-1.0, ops=[(1, True), (2, False)]),
 FermionTerm(coeff=-1.0, ops=[(2, True), (1, False)]),
 FermionTerm(coeff=-1.0, ops=[(2, True), (3, False)]),
 FermionTerm(coeff=-1.0, ops=[(3, True), (2, False)])]

In [ ]:
fermion_op = FermionOperator(hopping_terms, num_modes)
print(f"{len(fermion_op.terms)} FermionTerms over {fermion_op.num_modes} modes -- no qubits yet.")
fermion_op.terms

## 2. An encoding: Jordan-Wigner, interleaved order

`FermionicEncoding` is a value -- which map (`"jordan_wigner"`, for now) and which qubit each mode lands on (`mode_order[qubit] = mode`). It carries no reference to any particular Hamiltonian; the same encoding can be reused for any operator over the same `num_modes`.

"Interleaved" here just means a non-trivial permutation -- unlike `FermiHubbard1D`'s own `interleaved_mode_order` (which encodes a specific spin-pairing convention), this is a generic example: swap each adjacent pair of modes.

In [3]:
# qubit 0 <- mode 1, qubit 1 <- mode 0, qubit 2 <- mode 3, qubit 3 <- mode 2
interleaved_order = tuple(m for pair in zip(range(1, num_modes, 2), range(0, num_modes, 2)) for m in pair)
encoding = FermionicEncoding("jordan_wigner", num_modes=num_modes, mode_order=interleaved_order)
encoding

FermionicEncoding(name='jordan_wigner', num_modes=4, mode_order=(1, 0, 3, 2))

## 3. Apply the encoding: the result is just Paulis

`.encode()` is the *only* qubit-producing method in this whole module. Everything above (`FermionTerm`, `FermionOperator`) never mentioned a qubit; the object below is a plain `Hamiltonian` -- indistinguishable from one you'd get by hand-typing a `PauliSum`.

In [4]:
qubit_ham = fermion_op.encode(encoding)
print(type(qubit_ham).__name__)  # a plain Hamiltonian -- no fermionic residue
observable = qubit_ham.as_observable()
print(type(observable).__name__)
observable

Hamiltonian
PauliSum


((-0.5+0j))*XXII + ((-0.5+0j))*YYII + ((-0.5+0j))*YZZY + ((-0.5+0j))*XZZX + ((-0.5+0j))*IIXX + ((-0.5+0j))*IIYY

## 4. `.as_matrix()` gives a matrix on the correct number of qubits

In [5]:
H = qubit_ham.as_matrix()
print("shape:", H.shape, "expected:", (2**encoding.num_qubits, 2**encoding.num_qubits))
assert H.shape == (2**encoding.num_qubits, 2**encoding.num_qubits)
assert np.allclose(H, H.conj().T)  # Hermitian, as guaranteed by jordan_wigner's pairing
print("Hermitian: OK")

shape: (16, 16) expected: (16, 16)
Hermitian: OK


## 5. `.ground_state()` gives a `Statevector`

Ordinary exact diagonalization -- `Hamiltonian.ground_state()` doesn't know or care that this particular `Hamiltonian` came from encoding a `FermionOperator`.

In [6]:
gs = qubit_ham.ground_state()
print(type(gs).__name__)
gs

Statevector


0.2236|0011⟩ + -0.5000|0101⟩ + -0.4472|0110⟩ + -0.4472|1001⟩ + -0.5000|1010⟩ + 0.2236|1100⟩

## 6. Create a Majorana bilinear as an observable

Same split as before: define the *physics* of the observable (`i·γ_a·γ_b`, a `MajoranaTerm`) with no reference to qubits yet.

In [7]:
majorana_obs = [majorana_bilinear(0, 1)]  # i * gamma_0 * gamma_1, on mode 0
majorana_obs

[MajoranaTerm(coeff=1j, ops=[0, 1])]

## 7. Apply the *same* encoding to the Majorana observable -- a separate step

This is the point of `FermionicEncoding`: the exact same `encoding` object used for the Hamiltonian above is reused here, so the observable and the ground state it's about to be evaluated on are guaranteed to agree on which qubit is which mode.

In [8]:
encoded_majorana_obs = encoding.encode_majorana_terms(majorana_obs)
print(type(encoded_majorana_obs).__name__)
encoded_majorana_obs

PauliSum


((-1+0j))*IZII

## 8. Expectation value of the encoded Majorana operator in the ground state

`encoded_majorana_obs` is a plain `PauliSum` (an `Observable`), so `Statevector.expectation` handles it exactly as it would any hand-written Pauli observable -- no new backend code anywhere.

In [9]:
value = gs.expectation(encoded_majorana_obs)
print(f"<i*gamma_0*gamma_1> in the ground state = {value:.6f}")
assert abs(value.imag) < 1e-10  # Hermitian observable -> real expectation value

<i*gamma_0*gamma_1> in the ground state = -0.000000+0.000000j


## 9. A spinless dimer: single fermion -> symmetric superposition

The simplest possible check that all of this reproduces textbook physics: two modes, pure hopping, no interaction. In the one-particle sector the Hamiltonian is just the 2x2 hopping matrix `[[0, -t], [-t, 0]]`, whose ground state (for `t > 0`) is the *symmetric* (bonding) combination of "fermion on site 0" and "fermion on site 1", with energy `-t`.

In [10]:
t = 1.0
dimer = FermionOperator(hopping(0, 1, -t), num_modes=2)
dimer_encoding = FermionicEncoding("jordan_wigner", num_modes=2)  # default (identity) mode_order

dimer_ham = dimer.encode(dimer_encoding)
E0 = np.linalg.eigvalsh(dimer_ham.as_matrix())[0]
print(f"ground energy = {E0:.6f}  (expected -t = {-t:.6f})")
assert np.isclose(E0, -t)

dimer_gs = dimer_ham.ground_state(fixed_particle_number=1)
amplitudes = dimer_gs.to_array()
print("amplitudes:", amplitudes)
occupied = [i for i, amp in enumerate(amplitudes) if abs(amp) > 1e-9]
print("nonzero (single-occupation) basis states:", occupied)
assert len(occupied) == 2
assert np.isclose(abs(amplitudes[occupied[0]]), abs(amplitudes[occupied[1]]))
assert np.isclose(amplitudes[occupied[0]] * amplitudes[occupied[1]].conj(), 0.5)  # same sign -> symmetric
print("Equal-magnitude, same-sign amplitudes on the two single-occupation states: symmetric superposition confirmed.")

ground energy = -1.000000  (expected -t = -1.000000)
amplitudes: [ 0.    +0.j -0.7071-0.j -0.7071-0.j  0.    +0.j]
nonzero (single-occupation) basis states: [1, 2]
Equal-magnitude, same-sign amplitudes on the two single-occupation states: symmetric superposition confirmed.


## FermiHubbard1D: the same pattern, a real model

`FermiHubbard1D` is a `FermionOperator` subclass -- bare until encoded. "stacked" and "interleaved" (see `docs/jw_fermi_hubbard_1d.md`) are no longer constructor arguments; they're just two different `FermionicEncoding`s applied to the *same* bare model, and (per `docs/jw_fermi_hubbard_1d.md` eq. 20) must give the identical spectrum.

In [11]:
num_fermions = 3
fh = FermiHubbard1D(num_fermions, t=1.0, U=2.0, mu=0.5)  # bare -- no ordering argument at all

stacked = FermionicEncoding("jordan_wigner", num_modes=fh.num_modes)
interleaved = FermionicEncoding(
    "jordan_wigner", num_modes=fh.num_modes, mode_order=interleaved_mode_order(num_fermions)
)

E_stacked = np.linalg.eigvalsh(fh.encode(stacked).as_matrix())[0]
E_interleaved = np.linalg.eigvalsh(fh.encode(interleaved).as_matrix())[0]
print(f"stacked E0     = {E_stacked:.10f}")
print(f"interleaved E0 = {E_interleaved:.10f}")
assert abs(E_stacked - E_interleaved) < 1e-10

stacked E0     = -1.2794523158
interleaved E0 = -1.2794523158


### Trotter circuits, and the gate-fusion caveat

`FermiHubbard1D.encode()` reattaches its own Trotter-layer structure (which bonds mutually commute), so `trotter_circuit()` works -- but per-`FermionTerm` flattening means the old fused single onsite gate is now 3 separate gates. `trotter_circuit()` warns about this (once per call), without blocking or breaking anything.

In [12]:
import warnings
from scipy.linalg import expm

qubit_fh = fh.encode(stacked)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    circ = qubit_fh.trotter_circuit(dt=0.01, order=2)
    fusion_warnings = [w for w in caught if "gate-fusion" in str(w.message)]
    print(f"gate-fusion warnings raised: {len(fusion_warnings)}")
    if fusion_warnings:
        print(f"  -> {fusion_warnings[0].message}")

trotter_err = np.linalg.norm(circ.to_matrix() - expm(-1j * 0.01 * qubit_fh.as_matrix()), ord="fro")
print(f"Trotter-vs-exact Frobenius error at dt=0.01: {trotter_err:.2e}  (correctness unaffected by the warning)")

gate-fusion warnings raised: 1
  -> This Hamiltonian's terms were produced by a generic fermion-to-qubit encoding with no gate-fusion optimization pass -- the Trotter circuit may use more gates than necessary (e.g. commuting diagonal terms on the same qubits are not merged into one gate). Correctness and Trotter error scaling are unaffected.
Trotter-vs-exact Frobenius error at dt=0.01: 4.48e-06  (correctness unaffected by the warning)


/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: divide by zero encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: overflow encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: invalid value encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: divide by zero encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: overflow encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: invalid value encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T


## Summary

- `FermionOperator`/`MajoranaOperator` are pure algebra: no qubits, no encoding, until `.encode(...)` is called.
- `FermionicEncoding` is a value (map name + `mode_order`) built once and reused for both a Hamiltonian and any observable evaluated on its ground state -- the two can't silently drift onto different encodings.
- `.encode()` on either a Hamiltonian *or* an observable returns exactly the ordinary qubit-space types (`Hamiltonian`, `PauliSum`) already used everywhere else in the codebase -- nothing downstream needs to know fermions were ever involved.
- `jordan_wigner` guarantees every emitted `HamiltonianTerm` is Hermitian by auto-pairing non-self-adjoint terms (e.g. the two halves of a hopping bond); it raises `ValueError` if a term has no partner, rather than silently handing `Hamiltonian.trotter_circuit()` a non-unitary gate.
- What this trades away: the old hand-fused single onsite gate is now 3 separate (still-correct, still-commuting) gates -- flagged by a warning on `trotter_circuit()`, not hidden.